# How Many Generations Does Relatedness-Driven Similarity Last? (vem 1uM, 144hrs Pre-Treatment)

**Prototype.** `01_relatedness.ipynb` found that sisters (degree 1: common parent) and cousins (degree 2: common grandparent) are both more morphologically similar than random, for both modalities and both the control and treatment ranges. This notebook asks the natural follow-up: how far does that similarity reach - does it still hold for second cousins (degree 3: common great-grandparent), third cousins (degree 4), and so on, or does it fade out after a few generations?

`shared/lineage_tree.py`'s `find_relative_pairs(segments, degree)` generalizes `find_sister_pairs`/`find_cousin_pairs` to any `degree` (1 = sisters, 2 = cousins, 3 = second cousins, ...). For each `degree`, this notebook runs the exact same shared-null permutation test as `01_relatedness.ipynb`, once per (modality, range) - **4 lines total on one plot**: x-axis is `degree`, y-axis is `-log10(p-value)` (higher = more significant), with a horizontal line at `p = 0.05` so each line's crossing point is visible directly.

**The real constraint is sample size, not the method.** The number of degree-`d` pairs whose BOTH sides fall inside a 5-frame window shrinks fast as `d` grows - especially under treatment, where persisters/dying cells barely divide. A line stops because it ran out of qualifying pairs (printed, and visible as the line simply ending) is a *different* conclusion from a line that keeps going but crosses above `p = 0.05` (relatedness-driven similarity genuinely fades) - the printed `n_pairs` per point is what tells these apart, so read it alongside the plot rather than just the crossing point.

Requires `main.py` to have already been run (for `selected_metadata.csv` and `{cell,nuc}_qgw_075P_dists.csv` under `output/cajal_output/`).

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/05_vem_144hr_lineage_relatedness/notebooks/), so the analysis
# dir - and config.py - is one level up, and the repo root - where
# shared/ lives - is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR are relative to the analysis
# directory, not this notebook's own cwd - re-anchor them to an absolute
# path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.cajal.embed import load_distance_matrix
from shared.load_data import parse_label
from shared.lineage_tree import build_segments, find_relative_pairs, real_index_pairs, pool_in_range
from shared.pair_similarity import pair_distance_permutation_test

## Method

1. **Reconstruct every lineage tree** once (same as `01_relatedness.ipynb`, from the FULL metadata in frames 1-`config.TREATMENT_END_FRAME`, not just the CAJAL-sampled cells).
2. **For each modality** (cell, nucleus), load its real distance matrix once.
3. **For `degree` = 1, 2, 3, ...**: find every degree-`degree` relative pair across every tree (`find_relative_pairs`) - this step doesn't depend on modality or frame range, so it's done once per degree, not once per (modality, range). Then, separately for each (modality, range):
   - Restrict to whichever of each pair's own observations fall in that range AND survived CAJAL subsampling (`real_index_pairs`, exactly as in `01_relatedness.ipynb`) - every surviving combination counts as its own pair, not just one per relationship.
   - If this (modality, range) has zero surviving pairs at this degree, it has run out of data - stop extending that one line (the others may continue).
   - Otherwise, run the shared-null permutation test (`pair_distance_permutation_test`) against every CAJAL-embedded cell in that same range, using the pairs' own count as the draw size (unlike `01_relatedness.ipynb`, which calibrates sisters and cousins to a shared draw size so they share one null histogram - there's no such need here, since each `degree` gets its own point on this line plot, not an overlaid histogram).
4. **Plot**: one line per (modality, range) - 4 total - of `-log10(p-value)` vs. `degree`, with a horizontal line at `p = 0.05`.

In [ ]:
PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '02_relatedness_decay')
SVG_DIR = os.path.join(PLOTS_DIR, 'svgs')
os.makedirs(SVG_DIR, exist_ok=True)

MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}
MODALITY_COLORS = {'cell': 'tab:blue', 'nuc': 'tab:orange'}
RANGE_LINESTYLES = {'control': '-', 'treatment': '--'}

# same (start, end) frame ranges as 01_relatedness.ipynb
FRAME_RANGES = {
    'control':   (1, config.CONTROL_END_FRAME),
    'treatment': (config.TREATMENT_START_FRAME, config.TREATMENT_END_FRAME),
}

# a safety cap, not expected to actually bind - find_relative_pairs
# itself (independent of modality/range) is what actually stops the
# sweep in practice, as soon as NO tree anywhere has divided this many
# times yet; this just guards against an unexpectedly deep tree
MAX_DEGREE = 10

# below this many surviving pairs, a point's p-value is too noisy to
# read as a real trend - flagged in the printed summary, not hidden
MIN_RELIABLE_PAIRS = 5

## Build the lineage tree and load each modality's distance matrix (once)

In [ ]:
full_metadata = pd.read_csv(config.metadata_csv_path())

# tree-building uses the FULL metadata (every tracked cell, has_mask
# True or False) - has_mask only reflects whether a usable mask crop
# could be extracted, not whether the cell/lineage tracking itself
# exists. Filtering to has_mask=True here would drop intermediate rows
# a later frame's pred_node_id still needs, breaking chains that are
# otherwise intact.
full_metadata = full_metadata[full_metadata['frame'] <= config.TREATMENT_END_FRAME].reset_index(drop=True)

segments = build_segments(full_metadata)
print(f"Built {len(segments)} segment(s) across every lineage tree in frames 1-{config.TREATMENT_END_FRAME}")

modality_data = {}
for modality in ['cell', 'nuc']:
    dist_csv = os.path.join(config.CAJAL_OUTPUT_DIR, f'{modality}_qgw_{config.N_POINTS:03d}P_dists.csv')
    labels, dist_matrix = load_distance_matrix(dist_csv)
    label_to_index = {}
    for i, label in enumerate(labels):
        _, frame, pixel_id = parse_label(label)
        label_to_index[(frame, pixel_id)] = i

    modality_data[modality] = {
        'dist_matrix': dist_matrix,
        'label_to_index': label_to_index,
        # the null's population is fixed per (modality, range) - it
        # does NOT shrink with degree, only the real pairs being
        #  tested against it do
        'pool': {range_name: pool_in_range(label_to_index, frame_range) for range_name, frame_range in FRAME_RANGES.items()},
    }
    print(f"[{modality}] {len(labels)} CAJAL-embedded cell(s) total")

## Sweep: degree = 1, 2, 3, ... for each (modality, range)

In [ ]:
results = []
boot_inputs = {}   # (modality, range_name, degree) -> {'real_distances', 'null_means'}, for the bootstrap CI below
combos = [(modality, range_name) for modality in ['cell', 'nuc'] for range_name in FRAME_RANGES]
active = {combo: True for combo in combos}

for degree in range(1, MAX_DEGREE + 1):
    topology_pairs = find_relative_pairs(segments, degree)
    if not topology_pairs:
        print(f"degree {degree}: no lineage anywhere has divided this many times yet - stopping every line")
        break

    for modality, range_name in combos:
        if not active[(modality, range_name)]:
            continue

        data = modality_data[modality]
        frame_range = FRAME_RANGES[range_name]
        real_idx = real_index_pairs(topology_pairs, frame_range, data['label_to_index'])

        if len(real_idx) == 0:
            active[(modality, range_name)] = False
            print(f"  [{modality}/{range_name}] degree {degree}: 0 surviving pairs - stopping this line")
            continue

        mean_distance, null_means, p_value = pair_distance_permutation_test(
            data['dist_matrix'], data['pool'][range_name], real_idx, len(real_idx),
            n_permutations=config.N_PERMUTATIONS, seed=config.TEST_SEED,
        )
        flag = '  <-- fewer than MIN_RELIABLE_PAIRS, read with caution' if len(real_idx) < MIN_RELIABLE_PAIRS else ''
        print(f"  [{modality}/{range_name}] degree {degree}: {len(real_idx)} pair(s), "
              f"mean distance = {mean_distance:.4f}, p = {p_value:.4f}{flag}")

        results.append({
            'modality': modality, 'frame_range': range_name, 'degree': degree,
            'n_pairs': len(real_idx), 'mean_distance': mean_distance, 'p_value': p_value,
        })
        boot_inputs[(modality, range_name, degree)] = {
            'real_distances': np.array([data['dist_matrix'][i, j] for i, j in real_idx]),
            'null_means': null_means,
        }

    if not any(active.values()):
        break

results_df = pd.DataFrame(results)
print(f"\n{len(results_df)} (modality, range, degree) point(s) total")

## Plot: significance vs. relatedness degree, all 4 (modality, range) combinations on one plot

Each point's error bar is a 95% CI on its p-value from **bootstrap-resampling the real pairs themselves**: resample `n_pairs` real distances with replacement many times, recompute the resampled mean, and read off where that mean falls against the SAME (already-computed) null distribution each time - this directly answers "how much would this result change if we'd happened to observe a different set of `n_pairs` real relatives instead of these specific ones". It is **not** the permutation test's own Monte Carlo resolution (how finely `n_permutations` random draws can resolve a p-value) - that's a different, independent source of noise, and it does not get wider just because `n_pairs` is small. The bootstrap CI does: a point built from 1-2 real pairs will show a much wider interval than one built from hundreds, because a handful of observations just don't pin down a mean very precisely, however many permutations the null itself used.

A single real pair (`n_pairs == 1`) can't be bootstrapped at all - there's no variability to resample from one observation - so those points are drawn without an error bar and labeled `n=1` directly, rather than showing a misleadingly narrow (or fabricated) interval.

Wherever even the WORST-case resample still beat every null permutation draw (bootstrap `p_ci_lo == 0`), there's no bound on how small the true value could be - only a floor on how confidently this test can tell. The point is plotted at its p-value's own position (falling back to the CI's finite lower-significance bound only when the point estimate itself hit the floor, `p_value == 0`), with its real downward error drawn normally - a strongly significant point can still have a wide-open upward side - and a small arrow marks that the upward side is open-ended rather than a hard bound.

In [ ]:
def bootstrap_p_ci(real_distances, null_means, n_boot=2000, alpha=0.05, seed=0):
    '''
    95% CI on the TRUE p-value, from resampling the REAL pairs
    themselves - answers "how much would this result change if we'd
    happened to observe a different set of n_pairs real relatives
    instead of these specific ones", which is the uncertainty that
    actually should (and does) grow as n_pairs shrinks. This is NOT
    the permutation test's own Monte Carlo resolution (how finely
    n_permutations random draws can resolve a p-value) - that's a
    separate, independent source of noise, fixed by n_permutations
    alone, which does not get wider just because n_pairs is small.

    Resamples real_distances (with replacement) n_boot times, takes
    each resample's mean, and reads off its p-value against the SAME
    (already-computed) null_means every time - no new permutations are
    run here, only the real side of the comparison changes.

    Inputs:
        real_distances: the n_pairs real pairwise distances this
                        degree's point was computed from (same ones
                        pair_distance_permutation_test averaged for its
                        own observed_mean_distance)
        null_means:     from the SAME pair_distance_permutation_test call
        n_boot:         number of bootstrap resamples
        alpha:          1 - confidence level (0.05 -> 95% CI)
        seed:           RNG seed, for reproducibility
    Outputs:
        p_lo, p_hi: 95% CI bounds on the true p-value, or (nan, nan) if
                   n_pairs < 2 - there's no variability to resample from
                   a single observation, so no CI can be estimated at all
    '''
    real_distances = np.asarray(real_distances)
    n_pairs = len(real_distances)
    if n_pairs < 2:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    boot_means = rng.choice(real_distances, size=(n_boot, n_pairs), replace=True).mean(axis=1)
    boot_p = (null_means[None, :] <= boot_means[:, None]).mean(axis=1)
    return float(np.quantile(boot_p, alpha / 2)), float(np.quantile(boot_p, 1 - alpha / 2))


ci_rows = []
for _, row in results_df.iterrows():
    inputs = boot_inputs[(row['modality'], row['frame_range'], row['degree'])]
    p_lo, p_hi = bootstrap_p_ci(inputs['real_distances'], inputs['null_means'])
    ci_rows.append({'p_ci_lo': p_lo, 'p_ci_hi': p_hi})
results_df = pd.concat([results_df, pd.DataFrame(ci_rows)], axis=1)

P_FLOOR = 1 / config.N_PERMUTATIONS
fig, ax = plt.subplots(figsize=(7.5, 5.5))

for (modality, range_name), group in results_df.groupby(['modality', 'frame_range']):
    group = group.sort_values('degree')
    degree = group['degree'].to_numpy()
    p      = group['p_value'].to_numpy()
    p_lo   = group['p_ci_lo'].to_numpy()
    p_hi   = group['p_ci_hi'].to_numpy()
    has_ci = ~np.isnan(p_lo)   # False only for n_pairs == 1 points

    # the point itself: -log10(p), except wherever that's undefined
    # (p == 0) - there, fall back to the CI's own finite
    # lower-significance bound (-log10(p_hi)) if a CI exists, or the
    # test's bare resolution floor if it doesn't (n_pairs == 1, nothing
    # to bootstrap)
    fallback_p = np.where(has_ci, p_hi, P_FLOOR)
    fallback_p = np.where(fallback_p == 0, P_FLOOR, fallback_p)   # p_hi == 0 only when even the worst-case resample hit the floor too
    p_for_y = np.where(p == 0, fallback_p, p)
    y = -np.log10(p_for_y)

    # "less significant" error - always finite where a CI exists, 0 (no bar) where it doesn't
    p_hi_safe = np.where(p_hi == 0, P_FLOOR, p_hi)
    y_err_lo = np.where(has_ci, y - (-np.log10(p_hi_safe)), 0.0)

    # "more significant" error - open-ended wherever even the
    # worst-case resample beat every null draw (p_lo == 0): drawn as a
    # fixed-length arrow (lolims) instead of a real bound
    open_upper = has_ci & (p_lo == 0)
    p_lo_safe = np.where(p_lo == 0, 1.0, p_lo)   # placeholder, unused wherever p_lo == 0
    y_err_hi = np.where(open_upper, 0.4, np.where(has_ci, -np.log10(p_lo_safe) - y, 0.0))

    # plain two-sided error bar first, for every point - NOT matplotlib's
    # lolims (which, it turns out, suppresses the entire LOWER error too
    # for a point marked as a lower limit, not just the upper cap's
    # style - the wrong semantics for "normal downward error, but the
    # upward side is open-ended"). The open-ended upward side is instead
    # drawn as a separate arrow annotation, decoupled from the real
    # (and often substantial - e.g. a point can be strongly significant
    # while ITS OWN upward uncertainty is still unbounded) downward error.
    color = MODALITY_COLORS[modality]
    ax.errorbar(
        degree, y, yerr=[y_err_lo, y_err_hi], capsize=3,
        color=color, linestyle=RANGE_LINESTYLES[range_name], marker='o',
        label=f'{MODALITY_NAMES[modality]}, {range_name}',
    )
    for d, yy, yh in zip(degree[open_upper], y[open_upper], y_err_hi[open_upper]):
        ax.annotate('', xy=(d, yy + yh + 0.08), xytext=(d, yy + yh - 0.05),
                     arrowprops=dict(arrowstyle='-|>', color=color, lw=1.5))
    for d, yy in zip(degree[~has_ci], y[~has_ci]):
        ax.annotate('n=1', (d, yy), textcoords='offset points', xytext=(6, 4), fontsize=8, color=color)

threshold = -np.log10(0.05)
ax.axhline(threshold, color='grey', linestyle=':', linewidth=1.5)
ax.text(results_df['degree'].max(), threshold, '  p = 0.05', va='center', ha='left', color='grey')

ax.set_xlabel('relatedness degree (1 = sisters, 2 = cousins, 3 = second cousins, ...)')
ax.set_ylabel('-log10(p-value)\n(higher = more significantly more similar than a random pair)')
ax.set_xticks(sorted(results_df['degree'].unique()))
ax.set_title("How far does relatedness-driven similarity reach?\n(error bars: 95% CI from bootstrapping the real pairs)")
ax.legend()
fig.tight_layout()

fig.savefig(os.path.join(PLOTS_DIR, 'relatedness_decay.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(SVG_DIR, 'relatedness_decay.svg'), bbox_inches='tight')
display(fig)   # show it in the notebook's own output, not just save it to disk
plt.close(fig)
print(f"Saved to {PLOTS_DIR}")

## Where does each line cross p = 0.05?

The first degree, per (modality, range), where `p_value >= 0.05` - i.e. where relatedness-driven similarity stops being distinguishable from chance. A combo that ran out of pairs before ever crossing is reported separately: that's "not enough data to tell", not "similarity never fades".

In [ ]:
for modality, range_name in combos:
    group = results_df[(results_df['modality'] == modality) & (results_df['frame_range'] == range_name)].sort_values('degree')
    label = f'{MODALITY_NAMES[modality]}, {range_name}'
    if group.empty:
        print(f"{label}: no degree had any surviving pairs at all")
        continue

    lost = group[group['p_value'] >= 0.05]
    max_degree_reached = group['degree'].max()
    ran_out = not active[(modality, range_name)]

    if not lost.empty:
        crossing_degree = int(lost['degree'].iloc[0])
        n_pairs = int(lost['n_pairs'].iloc[0])
        p_lo, p_hi = lost['p_ci_lo'].iloc[0], lost['p_ci_hi'].iloc[0]
        ci_text = 'no CI possible (n_pairs == 1)' if np.isnan(p_lo) else f'95% CI for p: [{p_lo:.4f}, {p_hi:.4f}]'
        print(f"{label}: crosses p = 0.05 at degree {crossing_degree} (n_pairs = {n_pairs}, {ci_text})")
    elif ran_out:
        print(f"{label}: still significant through degree {max_degree_reached}, but ran out of pairs before crossing - "
              f"not enough data to say whether it would eventually cross")
    else:
        print(f"{label}: still significant through degree {max_degree_reached} (the sweep's own cap) - never crossed")

## Export plot data

One row per (modality, frame_range, degree): `n_pairs`, `mean_distance`, `p_value`, `p_ci_lo`, `p_ci_hi` (the bootstrap CI - `NaN` wherever `n_pairs == 1`) - everything needed to redraw the plot above (including its error bars) or re-derive the crossing points, without rerunning the permutation tests or the bootstrap.

In [ ]:
plot_data_path = os.path.join(PLOTS_DIR, 'plot_data.csv')
results_df.to_csv(plot_data_path, index=False)
print(f"Saved plot data ({len(results_df)} rows) to {plot_data_path}")